# 02 — Demand Forecasting: XGBoost Two-Stage Model + Shopping List

This notebook goes from preprocessed data to a per-facility shopping list for the upcoming week.

**Model**: Per-demand-class XGBoost with hard-threshold gating. Each demand class gets its own two-stage model — a classifier (will there be demand?) and a regressor (how much?). If the predicted probability exceeds a per-class threshold, we output the regressor's prediction; otherwise 0. Thresholds are tuned on the validation set using an asymmetric loss that penalises under-prediction 3x more than over-prediction.

**Rare pairs** (not enough history for ML) use a simple heuristic: `mean_nonzero × nonzero_rate`.

**Inputs**: `panel_weekly.parquet`, `unified_features.parquet` (from Notebook 01)  
**Output**: `facility_shopping_list_next_week.csv`

---
## Part A — Model Training & Validation

### Step 0 — Imports and Data Loading

In [1]:
from pathlib import Path
import time
import warnings
import numpy as np
import pandas as pd
import xgboost as xgb
from google.colab import drive, files
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings('ignore')
pd.set_option('display.width', 140)
pd.set_option('display.max_columns', 60)

drive.mount('/content/drive')

t0 = time.time()

Mounted at /content/drive


In [2]:
# ── Set your Google Drive data folder path here (same as Notebook 01) ──
DATA_DIR = Path('/content/drive/MyDrive/data')
PAIR_COLS = ['NEST_NAME', 'HEALTH_FACILITY_NAME', 'PRODUCT_NAME']

panel = pd.read_parquet(DATA_DIR / 'panel_weekly.parquet')
uni   = pd.read_parquet(DATA_DIR / 'unified_features.parquet')

print(f'panel_weekly shape:       {panel.shape}')
print(f'unified_features shape:   {uni.shape}')
print(f'\nPer-nest x split row counts (unified_features):')
print(uni.groupby(['NEST_NAME', 'split']).size().unstack(fill_value=0))

panel_weekly shape:       (4091010, 10)
unified_features shape:   (687678, 48)

Per-nest x split row counts (unified_features):
split         test   train  valid
NEST_NAME                        
GH1 Omenako   4240   85648   4240
GH3 Vobsi     9530  192506   9530
RW1 Muhanga  17770  209686  17770
RW2 Kayonza   9910  116938   9910


### Step 1 — Prepare Train / Validation / Test Splits

In [3]:
FEATURES = [
    'week_of_year', 'month', 'quarter', 'week_sin', 'week_cos', 'rainy_season',
    'lag_1', 'lag_2', 'lag_3', 'lag_4', 'lag_8', 'lag_12',
    'roll_mean_4', 'roll_std_4', 'roll_max_4', 'roll_nz_rate_4',
    'roll_mean_8', 'roll_std_8', 'roll_max_8', 'roll_nz_rate_8',
    'roll_mean_13', 'roll_std_13', 'roll_max_13', 'roll_nz_rate_13',
    'weeks_since_last_demand', 'last_nonzero_units',
    'historical_nonzero_rate', 'cumulative_active_weeks',
    'facility_hist_mean', 'facility_hist_nz_rate',
    'product_hist_mean',  'product_hist_nz_rate',
    'fp_hist_mean',       'fp_hist_nz_rate',
    'adi', 'cv2', 'mean_nonzero', 'nonzero_weeks',
    'HEALTH_FACILITY_NAME_enc', 'PRODUCT_NAME_enc',
]

train = uni[uni['split'] == 'train'].dropna(subset=['lag_1']).reset_index(drop=True)
val   = uni[uni['split'] == 'valid'].reset_index(drop=True)
test  = uni[uni['split'] == 'test' ].reset_index(drop=True)

print(f'Rows  train={len(train):,}  valid={len(val):,}  test={len(test):,}')
print(f'Train zero rate: {(train["weekly_units"] == 0).mean():.1%}')

Rows  train=596,488  valid=41,450  test=41,450
Train zero rate: 82.6%


### Step 2 — Per-Demand-Class Feature Sets

Different demand patterns need different signals — e.g., Smooth pairs rely on recent lags while Intermittent pairs depend on timing features. Each class gets a tailored feature subset.

In [4]:
DEMAND_CLASSES = ['Smooth', 'Erratic', 'Intermittent', 'Lumpy']

FEATURE_SETS = {
    'Smooth': [
        'week_of_year', 'month', 'quarter', 'week_sin', 'week_cos', 'rainy_season',
        'lag_1', 'lag_2', 'lag_3', 'lag_4',
        'roll_mean_4', 'roll_std_4', 'roll_max_4', 'roll_nz_rate_4',
        'roll_mean_8', 'roll_std_8', 'roll_max_8', 'roll_nz_rate_8',
        'historical_nonzero_rate', 'cumulative_active_weeks',
        'facility_hist_mean', 'facility_hist_nz_rate',
        'product_hist_mean',  'product_hist_nz_rate',
        'fp_hist_mean',       'fp_hist_nz_rate',
        'adi', 'cv2', 'mean_nonzero', 'nonzero_weeks',
        'HEALTH_FACILITY_NAME_enc', 'PRODUCT_NAME_enc',
    ],
    'Erratic': [
        'week_of_year', 'month', 'quarter', 'week_sin', 'week_cos', 'rainy_season',
        'lag_1', 'lag_2', 'lag_3', 'lag_4', 'lag_8', 'lag_12',
        'roll_mean_4',  'roll_std_4',  'roll_max_4',  'roll_nz_rate_4',
        'roll_mean_8',  'roll_std_8',  'roll_max_8',  'roll_nz_rate_8',
        'roll_mean_13', 'roll_std_13', 'roll_max_13', 'roll_nz_rate_13',
        'weeks_since_last_demand', 'last_nonzero_units',
        'historical_nonzero_rate', 'cumulative_active_weeks',
        'facility_hist_mean', 'facility_hist_nz_rate',
        'product_hist_mean',  'product_hist_nz_rate',
        'fp_hist_mean',       'fp_hist_nz_rate',
        'adi', 'cv2', 'mean_nonzero', 'nonzero_weeks',
        'HEALTH_FACILITY_NAME_enc', 'PRODUCT_NAME_enc',
    ],
    'Intermittent': [
        'week_of_year', 'month', 'quarter', 'week_sin', 'week_cos', 'rainy_season',
        'lag_1', 'lag_2', 'lag_4', 'lag_8', 'lag_12',
        'roll_nz_rate_4', 'roll_max_4',  'roll_mean_4',
        'roll_nz_rate_8', 'roll_max_8',  'roll_mean_8',
        'roll_nz_rate_13','roll_max_13', 'roll_mean_13',
        'weeks_since_last_demand', 'last_nonzero_units',
        'historical_nonzero_rate', 'cumulative_active_weeks',
        'facility_hist_mean', 'facility_hist_nz_rate',
        'product_hist_mean',  'product_hist_nz_rate',
        'fp_hist_mean',       'fp_hist_nz_rate',
        'adi', 'cv2', 'mean_nonzero', 'nonzero_weeks',
        'HEALTH_FACILITY_NAME_enc', 'PRODUCT_NAME_enc',
    ],
    'Lumpy': [
        'week_of_year', 'month', 'quarter', 'week_sin', 'week_cos', 'rainy_season',
        'lag_1', 'lag_2', 'lag_4', 'lag_8', 'lag_12',
        'roll_nz_rate_4', 'roll_max_4',  'roll_std_4',  'roll_mean_4',
        'roll_nz_rate_8', 'roll_max_8',  'roll_std_8',  'roll_mean_8',
        'roll_nz_rate_13','roll_max_13', 'roll_std_13', 'roll_mean_13',
        'weeks_since_last_demand', 'last_nonzero_units',
        'historical_nonzero_rate', 'cumulative_active_weeks',
        'facility_hist_mean', 'facility_hist_nz_rate',
        'product_hist_mean',  'product_hist_nz_rate',
        'fp_hist_mean',       'fp_hist_nz_rate',
        'adi', 'cv2', 'mean_nonzero', 'nonzero_weeks',
        'HEALTH_FACILITY_NAME_enc', 'PRODUCT_NAME_enc',
    ],
}

print('Feature counts per class:', {dc: len(f) for dc, f in FEATURE_SETS.items()})
print('\nDemand class distribution (train):')
print(train['demand_class'].value_counts())

Feature counts per class: {'Smooth': 32, 'Erratic': 40, 'Intermittent': 36, 'Lumpy': 39}

Demand class distribution (train):
demand_class
Intermittent    459282
Lumpy           131116
Smooth            5510
Erratic            580
Name: count, dtype: int64


### Step 3 — Metrics and Model Config

Primary metric: **Asymmetric WAPE** — under-prediction penalised 3x more than over-prediction, because in medical supply chains, stock-outs are far worse than having extra inventory.

In [5]:
UNDER_PENALTY = 3.0
MIN_POS_FOR_STAGE2 = 30

XGB_COMMON = dict(
    n_estimators=400, learning_rate=0.05, max_depth=5,
    subsample=0.8, colsample_bytree=0.8, min_child_weight=5,
    random_state=42, n_jobs=-1,
)

def wape(y, p):
    y, p = np.asarray(y, float), np.asarray(p, float)
    return float(np.sum(np.abs(p - y)) / max(np.sum(np.abs(y)), 1e-9))

def mae(y, p):
    return float(np.mean(np.abs(np.asarray(p, float) - np.asarray(y, float))))

def asym_wape(y, p, penalty=UNDER_PENALTY):
    y, p = np.asarray(y, float), np.asarray(p, float)
    err = y - p
    loss = np.where(err > 0, penalty * err, -err)
    return float(np.sum(loss) / (penalty * max(np.sum(np.abs(y)), 1e-9)))

def upr_opr(y, p):
    y, p = np.asarray(y, float), np.asarray(p, float)
    total_act = max(float(np.sum(y)), 1e-9)
    return (float(np.sum(np.maximum(y - p, 0))) / total_act,
            float(np.sum(np.maximum(p - y, 0))) / total_act)

### Step 4 — Train Per-Class Models + Threshold Search

For each demand class: train Stage 1 (classifier) and Stage 2 (regressor on positives), then sweep thresholds on the validation set to minimise Asymmetric WAPE.

In [6]:
XGB_MODELS = {}

for dc in DEMAND_CLASSES:
    feats_dc = FEATURE_SETS[dc]
    tr_dc = train[train['demand_class'] == dc]
    va_dc = val[val['demand_class'] == dc]

    X_tr_dc = tr_dc[feats_dc].fillna(-999)
    y_tr_dc = tr_dc['weekly_units'].astype(float).reset_index(drop=True)
    X_va_dc = va_dc[feats_dc].fillna(-999)
    y_va_dc = va_dc['weekly_units'].astype(float).reset_index(drop=True)

    occ_tr_dc = (y_tr_dc > 0).astype(int)
    occ_va_dc = (y_va_dc > 0).astype(int)
    pos_mask_tr = (y_tr_dc > 0).to_numpy()
    pos_mask_va = (y_va_dc > 0).to_numpy()
    n_pos = int(pos_mask_tr.sum())
    n_neg = len(occ_tr_dc) - n_pos
    spw_dc = n_neg / max(n_pos, 1)

    print(f'\n{"═" * 60}')
    print(f'  {dc}  |  train={len(tr_dc):,}  val={len(va_dc):,}  '
          f'pos_train={n_pos:,}  features={len(feats_dc)}')
    print(f'{"═" * 60}')

    # ── Stage 1: Occurrence Classifier ──
    clf_dc = xgb.XGBClassifier(
        **XGB_COMMON,
        objective='binary:logistic', eval_metric=['logloss', 'aucpr'],
        scale_pos_weight=spw_dc,
        early_stopping_rounds=30, verbosity=0,
    )
    clf_dc.fit(X_tr_dc, occ_tr_dc, eval_set=[(X_va_dc, occ_va_dc)], verbose=False)
    print(f'  Stage-1 best iteration: {clf_dc.best_iteration}')

    # ── Stage 2: Magnitude Regressor (positive rows only) ──
    reg_dc = None
    if n_pos >= MIN_POS_FOR_STAGE2 and int(pos_mask_va.sum()) >= 5:
        reg_dc = xgb.XGBRegressor(
            **XGB_COMMON,
            objective='reg:squarederror', eval_metric='rmse',
            early_stopping_rounds=30, verbosity=0,
        )
        reg_dc.fit(
            X_tr_dc[pos_mask_tr], np.log1p(y_tr_dc[pos_mask_tr]),
            eval_set=[(X_va_dc[pos_mask_va], np.log1p(y_va_dc[pos_mask_va]))],
            verbose=False,
        )
        print(f'  Stage-2 best iteration: {reg_dc.best_iteration}')
    else:
        print(f'  Stage-2 skipped (only {n_pos} positive rows)')

    # ── Threshold Search on Validation ──
    best_thr = 0.35
    if reg_dc is not None:
        prob_va = clf_dc.predict_proba(X_va_dc)[:, 1]
        mag_va  = np.expm1(reg_dc.predict(X_va_dc)).clip(min=0)
        best_aw = float('inf')
        for thr in np.arange(0.20, 0.61, 0.025):
            hp = np.where(prob_va >= thr, mag_va, 0.0)
            aw = asym_wape(y_va_dc.to_numpy(), hp)
            if aw < best_aw:
                best_aw, best_thr = aw, round(float(thr), 3)
        print(f'  Optimal threshold: {best_thr:.3f}  (val AsymWAPE={best_aw:.4f})')

    XGB_MODELS[dc] = dict(clf=clf_dc, reg=reg_dc,
                          features=feats_dc, threshold=best_thr)

print(f'\nAll classes trained: {list(XGB_MODELS.keys())}')
print(f'elapsed={time.time()-t0:.1f}s')


════════════════════════════════════════════════════════════
  Smooth  |  train=5,510  val=475  pos_train=4,801  features=32
════════════════════════════════════════════════════════════
  Stage-1 best iteration: 35
  Stage-2 best iteration: 40
  Optimal threshold: 0.200  (val AsymWAPE=0.3772)

════════════════════════════════════════════════════════════
  Erratic  |  train=580  val=50  pos_train=490  features=40
════════════════════════════════════════════════════════════
  Stage-1 best iteration: 2
  Stage-2 best iteration: 1
  Optimal threshold: 0.200  (val AsymWAPE=0.5143)

════════════════════════════════════════════════════════════
  Intermittent  |  train=459,282  val=32,790  pos_train=75,650  features=36
════════════════════════════════════════════════════════════
  Stage-1 best iteration: 67
  Stage-2 best iteration: 78
  Optimal threshold: 0.600  (val AsymWAPE=0.9322)

════════════════════════════════════════════════════════════
  Lumpy  |  train=131,116  val=8,135  pos_train

### Step 5 — Evaluate on Test Set

In [7]:
pieces = []
for dc in DEMAND_CLASSES:
    model = XGB_MODELS[dc]
    te_dc = test[test['demand_class'] == dc].copy()
    if len(te_dc) == 0:
        continue
    feats_dc = model['features']
    clf_dc, reg_dc, thr_dc = model['clf'], model['reg'], model['threshold']
    X_te_dc = te_dc[feats_dc].fillna(-999)

    prob_te = clf_dc.predict_proba(X_te_dc)[:, 1]
    if reg_dc is not None:
        mag_te    = np.expm1(reg_dc.predict(X_te_dc)).clip(min=0)
        pred_hard = np.where(prob_te >= thr_dc, mag_te, 0.0)
    else:
        mag_te    = np.zeros(len(te_dc))
        pred_hard = np.zeros(len(te_dc))

    te_dc['y_pred']  = pred_hard
    te_dc['s1_prob'] = prob_te
    te_dc['s2_mag']  = mag_te
    pieces.append(te_dc)

xgb_test = pd.concat(pieces).sort_index()

# Per-nest test metrics
print('Test Performance — Per-Nest (Hard-Threshold Prediction):')
print(f'{"─" * 80}')
rows = []
for nest, g in xgb_test.groupby('NEST_NAME'):
    y = g['weekly_units'].to_numpy()
    p = g['y_pred'].to_numpy()
    u, o = upr_opr(y, p)
    rows.append({'Nest': nest, 'Rows': len(g),
                 'MAE': round(mae(y, p), 3),
                 'WAPE': round(wape(y, p), 4),
                 'AsymWAPE': round(asym_wape(y, p), 4),
                 'UPR': round(u, 4), 'OPR': round(o, 4)})
print(pd.DataFrame(rows).to_string(index=False))

# Overall metrics
y_all = xgb_test['weekly_units'].to_numpy()
p_all = xgb_test['y_pred'].to_numpy()
u_all, o_all = upr_opr(y_all, p_all)
print(f'\nOverall:  MAE={mae(y_all, p_all):.3f}  WAPE={wape(y_all, p_all):.4f}  '
      f'AsymWAPE={asym_wape(y_all, p_all):.4f}  UPR={u_all:.4f}  OPR={o_all:.4f}')

Test Performance — Per-Nest (Hard-Threshold Prediction):
────────────────────────────────────────────────────────────────────────────────
       Nest  Rows   MAE   WAPE  AsymWAPE    UPR    OPR
GH1 Omenako  4240 1.231 1.1196    0.9166 0.8152 0.3045
  GH3 Vobsi  9530 2.460 1.1577    0.9262 0.8105 0.3472
RW1 Muhanga 17770 4.754 1.1470    0.9621 0.8697 0.2773
RW2 Kayonza  9910 5.253 1.1715    0.9594 0.8534 0.3181

Overall:  MAE=3.985  WAPE=1.1552  AsymWAPE=0.9547  UPR=0.8545  OPR=0.3008


---
## Part B — Production Forecasting: Next-Week Shopping List

Now we retrain on **all** data (train + valid + test), recompute features using full history, score the upcoming week, and combine with the rare-pair heuristic to produce the final shopping list.

### Step 6 — Determine the Next Prediction Week per Nest

In [8]:
mod_panel = panel[panel['tier'] == 'modellable'][
    PAIR_COLS + ['week_start', 'weekly_units']].copy()
mod_panel = mod_panel.sort_values(PAIR_COLS + ['week_start']).reset_index(drop=True)

nest_next_week = (mod_panel.groupby('NEST_NAME')['week_start'].max()
                  .reset_index()
                  .assign(next_week=lambda d: d['week_start'] + pd.Timedelta(weeks=1))
                  [['NEST_NAME', 'next_week']])

print('Next prediction week per nest:')
print(nest_next_week.to_string(index=False))

Next prediction week per nest:
  NEST_NAME  next_week
GH1 Omenako 2026-02-16
  GH3 Vobsi 2026-02-16
RW1 Muhanga 2026-04-27
RW2 Kayonza 2026-04-27


### Step 7 — Append Next-Week Rows and Recompute Features

Add a placeholder row per modellable pair for the upcoming week, then rebuild all 40 features using the full historical series (same logic as Notebook 01, but now with all data).

In [9]:
# Append next-week placeholder rows
pair_to_nest = mod_panel[PAIR_COLS].drop_duplicates()
pair_to_nest = pair_to_nest.merge(nest_next_week, on='NEST_NAME', how='left')
extra_rows = pair_to_nest.rename(columns={'next_week': 'week_start'})
extra_rows['weekly_units'] = np.nan

prod_panel = pd.concat([mod_panel, extra_rows], ignore_index=True).sort_values(
    PAIR_COLS + ['week_start']).reset_index(drop=True)
print(f'Production panel rows: {len(prod_panel):,}')

Production panel rows: 695,968


In [10]:
f = prod_panel.copy()

# ── Calendar features ──
f['week_of_year'] = f['week_start'].dt.isocalendar().week.astype(int)
f['month']        = f['week_start'].dt.month
f['quarter']      = f['week_start'].dt.quarter
f['week_sin']     = np.sin(2 * np.pi * f['week_of_year'] / 52)
f['week_cos']     = np.cos(2 * np.pi * f['week_of_year'] / 52)
f['rainy_season'] = f['month'].between(4, 10).astype(int)

# ── Lags ──
f['units_filled'] = f['weekly_units'].fillna(0.0)
g_units = f.groupby(PAIR_COLS, sort=False)['units_filled']
for k in [1, 2, 3, 4, 8, 12]:
    f[f'lag_{k}'] = g_units.shift(k)

# ── Rolling features ──
shifted_units = f.groupby(PAIR_COLS, sort=False)['units_filled'].shift(1)
shifted_occ   = (shifted_units > 0).astype(float)
keys = [f[c] for c in PAIR_COLS]
for w in [4, 8, 13]:
    f[f'roll_mean_{w}'] = shifted_units.groupby(keys).transform(
        lambda s: s.rolling(w, min_periods=1).mean())
    f[f'roll_std_{w}'] = shifted_units.groupby(keys).transform(
        lambda s: s.rolling(w, min_periods=1).std().fillna(0))
    f[f'roll_max_{w}'] = shifted_units.groupby(keys).transform(
        lambda s: s.rolling(w, min_periods=1).max())
    f[f'roll_nz_rate_{w}'] = shifted_occ.groupby(keys).transform(
        lambda s: s.rolling(w, min_periods=1).mean())

print(f'Calendar + lag + rolling features done. elapsed={time.time()-t0:.1f}s')

Calendar + lag + rolling features done. elapsed=139.6s


In [11]:
# ── Intermittent-demand features ──
ws_all, lnz_all, hnz_all, ca_all = [], [], [], []
for _, g in f.groupby(PAIR_COLS, sort=False):
    units = g['units_filled'].to_numpy()
    n = len(units)
    ws  = np.full(n, np.nan)
    lnz = np.full(n, np.nan)
    hnz = np.full(n, np.nan)
    ca  = np.zeros(n)
    last_idx, last_val, cum = -1, np.nan, 0
    for i in range(n):
        if i > 0:
            hnz[i] = cum / i
            ca[i]  = cum
            if last_idx >= 0:
                ws[i] = i - last_idx
                lnz[i] = last_val
        if units[i] > 0:
            last_idx, last_val, cum = i, units[i], cum + 1
    ws_all.extend(ws)
    lnz_all.extend(lnz)
    hnz_all.extend(hnz)
    ca_all.extend(ca)

f['weeks_since_last_demand'] = ws_all
f['last_nonzero_units']      = lnz_all
f['historical_nonzero_rate'] = hnz_all
f['cumulative_active_weeks'] = ca_all
print(f'Intermittent features done. elapsed={time.time()-t0:.1f}s')

Intermittent features done. elapsed=148.1s


In [12]:
# ── Expanding history features (facility / product / pair level) ──
fac = (f.groupby(['NEST_NAME', 'HEALTH_FACILITY_NAME', 'week_start'], as_index=False)
        ['units_filled'].sum().rename(columns={'units_filled': 'fac_total'})
        .sort_values(['NEST_NAME', 'HEALTH_FACILITY_NAME', 'week_start']))
fac['fac_nz'] = (fac['fac_total'] > 0).astype(int)
gf = fac.groupby(['NEST_NAME', 'HEALTH_FACILITY_NAME'])
fac['facility_hist_mean']    = gf['fac_total'].transform(
    lambda s: s.shift(1).expanding().mean())
fac['facility_hist_nz_rate'] = gf['fac_nz'].transform(
    lambda s: s.shift(1).expanding().mean())
f = f.merge(fac[['NEST_NAME', 'HEALTH_FACILITY_NAME', 'week_start',
                 'facility_hist_mean', 'facility_hist_nz_rate']],
            on=['NEST_NAME', 'HEALTH_FACILITY_NAME', 'week_start'], how='left')

prod = (f.groupby(['NEST_NAME', 'PRODUCT_NAME', 'week_start'], as_index=False)
         ['units_filled'].sum().rename(columns={'units_filled': 'prod_total'})
         .sort_values(['NEST_NAME', 'PRODUCT_NAME', 'week_start']))
prod['prod_nz'] = (prod['prod_total'] > 0).astype(int)
gp = prod.groupby(['NEST_NAME', 'PRODUCT_NAME'])
prod['product_hist_mean']    = gp['prod_total'].transform(
    lambda s: s.shift(1).expanding().mean())
prod['product_hist_nz_rate'] = gp['prod_nz'].transform(
    lambda s: s.shift(1).expanding().mean())
f = f.merge(prod[['NEST_NAME', 'PRODUCT_NAME', 'week_start',
                  'product_hist_mean', 'product_hist_nz_rate']],
            on=['NEST_NAME', 'PRODUCT_NAME', 'week_start'], how='left')

f = f.sort_values(PAIR_COLS + ['week_start']).reset_index(drop=True)
g_pair = f.groupby(PAIR_COLS, sort=False)['units_filled']
f['fp_hist_mean'] = g_pair.transform(lambda s: s.shift(1).expanding().mean())
f['fp_nz_tmp']    = (f['units_filled'] > 0).astype(int)
f['fp_hist_nz_rate'] = f.groupby(PAIR_COLS, sort=False)['fp_nz_tmp'].transform(
    lambda s: s.shift(1).expanding().mean())
f = f.drop(columns=['fp_nz_tmp', 'units_filled'])

print(f'Expanding history done. elapsed={time.time()-t0:.1f}s')

Expanding history done. elapsed=163.2s


In [13]:
# ── Pair-level static features + encodings ──
ps = uni[PAIR_COLS + ['adi', 'cv2', 'mean_nonzero', 'nonzero_weeks']].drop_duplicates(PAIR_COLS)
f = f.merge(ps, on=PAIR_COLS, how='left')

fac_le = LabelEncoder().fit(uni['HEALTH_FACILITY_NAME'].astype(str))
prd_le = LabelEncoder().fit(uni['PRODUCT_NAME'].astype(str))
f['HEALTH_FACILITY_NAME_enc'] = fac_le.transform(f['HEALTH_FACILITY_NAME'].astype(str))
f['PRODUCT_NAME_enc']         = prd_le.transform(f['PRODUCT_NAME'].astype(str))

print(f'Production feature frame shape: {f.shape}')
print(f'\nNext-week rows per nest:')
print(f.merge(nest_next_week, left_on=['NEST_NAME', 'week_start'],
              right_on=['NEST_NAME', 'next_week'])
       .groupby('NEST_NAME').size())

Production feature frame shape: (695968, 45)

Next-week rows per nest:
NEST_NAME
GH1 Omenako     848
GH3 Vobsi      1906
RW1 Muhanga    3554
RW2 Kayonza    1982
dtype: int64


### Step 8 — Retrain on Full History and Score Next Week

Retrain per-class models on all data, reusing the thresholds from Step 4. Score each next-week row with its class-specific model.

In [14]:
f_with_next = f.merge(nest_next_week, on='NEST_NAME', how='left')
is_next = f_with_next['week_start'] == f_with_next['next_week']

full_train = f_with_next[~is_next].copy()
full_train['weekly_units'] = full_train['weekly_units'].fillna(0.0)
full_train = full_train.dropna(subset=['lag_1']).reset_index(drop=True)
next_rows  = f_with_next[is_next].copy()

dc_map = uni[PAIR_COLS + ['demand_class']].drop_duplicates(PAIR_COLS)
full_train = full_train.merge(dc_map, on=PAIR_COLS, how='left')
full_train['demand_class'] = full_train['demand_class'].fillna('Intermittent')
next_rows = next_rows.merge(dc_map, on=PAIR_COLS, how='left')
next_rows['demand_class'] = next_rows['demand_class'].fillna('Intermittent')

print(f'Full retrain rows: {len(full_train):,}')
print(f'Next-week rows to score: {len(next_rows):,}')

Full retrain rows: 679,388
Next-week rows to score: 8,290


In [15]:
XGB_PROD = dict(
    n_estimators=300, learning_rate=0.05, max_depth=5,
    subsample=0.8, colsample_bytree=0.8, min_child_weight=5,
    random_state=42, n_jobs=-1,
)

pred_pieces = []

for dc in DEMAND_CLASSES:
    feats_dc = FEATURE_SETS[dc]
    thr_dc   = XGB_MODELS[dc]['threshold']
    ft_dc = full_train[full_train['demand_class'] == dc].reset_index(drop=True)
    if len(ft_dc) == 0:
        print(f'  [{dc}] no rows, skipping')
        continue

    X_dc  = ft_dc[feats_dc].fillna(-999)
    y_dc  = ft_dc['weekly_units'].astype(float)
    occ_dc = (y_dc > 0).astype(int)
    pos_dc = (y_dc > 0).to_numpy()
    n_pos  = int(pos_dc.sum())
    spw_dc = float((occ_dc == 0).sum()) / max(n_pos, 1)

    # Stage 1: classifier
    clf_dc = xgb.XGBClassifier(**XGB_PROD,
                                objective='binary:logistic',
                                eval_metric=['logloss', 'aucpr'],
                                scale_pos_weight=spw_dc, verbosity=0)
    clf_dc.fit(X_dc, occ_dc)

    # Stage 2: regressor
    reg_dc = None
    if n_pos >= MIN_POS_FOR_STAGE2:
        reg_dc = xgb.XGBRegressor(**XGB_PROD,
                                   objective='reg:squarederror',
                                   eval_metric='rmse', verbosity=0)
        reg_dc.fit(X_dc[pos_dc], np.log1p(y_dc[pos_dc]))

    print(f'  [{dc}] trained on {len(ft_dc):,} rows  pos={n_pos:,}  '
          f'stage2={reg_dc is not None}  threshold={thr_dc}')

    # Score next-week rows for this demand class
    nr_dc = next_rows[next_rows['demand_class'] == dc].copy()
    if len(nr_dc) == 0:
        continue
    X_nr = nr_dc[feats_dc].fillna(-999)
    prob = clf_dc.predict_proba(X_nr)[:, 1]
    if reg_dc is not None:
        mag = np.expm1(reg_dc.predict(X_nr)).clip(min=0)
        nr_dc['predicted_units'] = np.where(prob >= thr_dc, mag, 0.0)
    else:
        mag = np.zeros(len(nr_dc))
        nr_dc['predicted_units'] = 0.0
    nr_dc['s1_prob'] = prob
    nr_dc['s2_mag']  = mag
    pred_pieces.append(nr_dc)

next_scored = pd.concat(pred_pieces).sort_index()
print(f'\nScored {len(next_scored):,} modellable next-week rows')

  [Smooth] trained on 6,460 rows  pos=5,581  stage2=True  threshold=0.2
  [Erratic] trained on 680 rows  pos=568  stage2=True  threshold=0.2
  [Intermittent] trained on 524,862 rows  pos=86,569  stage2=True  threshold=0.6
  [Lumpy] trained on 147,386 rows  pos=25,407  stage2=True  threshold=0.6

Scored 8,290 modellable next-week rows


### Step 9 — Heuristic for Rare Pairs

For pairs without enough history for ML: `predicted_units = mean_nonzero × nonzero_rate` — the long-run expected weekly demand.

In [16]:
rare_panel = panel[panel['tier'] == 'rare'].copy()
rare_stats = (rare_panel.groupby(PAIR_COLS, as_index=False)
              .agg(historical_mean_nonzero=('weekly_units',
                       lambda x: float(x[x > 0].mean()) if (x > 0).any() else 0.0),
                   historical_nonzero_rate=('weekly_units',
                       lambda x: float((x > 0).mean()))))

rare_stats['predicted_units'] = (
    rare_stats['historical_mean_nonzero'] * rare_stats['historical_nonzero_rate'])
rare_stats = rare_stats.merge(nest_next_week, on='NEST_NAME', how='left').rename(
    columns={'next_week': 'week_start'})
rare_stats['s1_prob'] = np.nan
rare_stats['s2_mag']  = np.nan

print(f'Heuristic predictions for {len(rare_stats):,} rare pairs')

Heuristic predictions for 40,066 rare pairs


### Step 10 — Assemble the Final Shopping List

Combine ML predictions (modellable) and heuristic (rare), sort by predicted demand within each facility.

In [17]:
ml_part = next_scored[PAIR_COLS + ['week_start', 'predicted_units',
                                    's1_prob', 's2_mag']].copy()
ml_part['tier']   = 'modellable'
ml_part['method'] = 'XGBoost_hard'

rare_part = rare_stats[PAIR_COLS + ['week_start', 'predicted_units',
                                    's1_prob', 's2_mag']].copy()
rare_part['tier']   = 'rare'
rare_part['method'] = 'historical_heuristic'

shopping = pd.concat([ml_part, rare_part], ignore_index=True)

dc_map_all = panel[PAIR_COLS + ['demand_class']].drop_duplicates(PAIR_COLS)
shopping = shopping.merge(dc_map_all, on=PAIR_COLS, how='left')

shopping['predicted_units_rounded'] = (
    shopping['predicted_units'].round().astype(int).clip(lower=0))

shopping = shopping.rename(columns={
    'NEST_NAME': 'nest',
    'HEALTH_FACILITY_NAME': 'facility',
    'PRODUCT_NAME': 'product',
})

shopping = shopping.sort_values(
    ['nest', 'facility', 'predicted_units'],
    ascending=[True, True, False])

out_cols = ['nest', 'facility', 'product', 'week_start', 'predicted_units',
            'predicted_units_rounded', 'tier', 'demand_class', 'method',
            's1_prob', 's2_mag']
shopping = shopping[out_cols].reset_index(drop=True)

shopping.to_csv(DATA_DIR / 'facility_shopping_list_next_week.csv', index=False)

print(f'Shopping list saved: {len(shopping):,} rows')
print(f'  modellable (ML):     {(shopping.tier=="modellable").sum():,}')
print(f'  rare (heuristic):    {(shopping.tier=="rare").sum():,}')
print(f'\nPer-nest counts:')
print(shopping.groupby('nest').size())

Shopping list saved: 48,356 rows
  modellable (ML):     8,290
  rare (heuristic):    40,066

Per-nest counts:
nest
GH1 Omenako     9296
GH3 Vobsi       8667
RW1 Muhanga    18930
RW2 Kayonza    11463
dtype: int64


### Step 11 — Preview: Sample Facility Shopping Lists

In [18]:
sample_facilities = (shopping[['nest', 'facility']].drop_duplicates()
                     .sample(min(4, shopping[['nest', 'facility']]
                                 .drop_duplicates().shape[0]), random_state=42))

for _, r in sample_facilities.iterrows():
    sub = (shopping[(shopping.nest == r['nest']) & (shopping.facility == r['facility'])]
           .head(10)[['product', 'predicted_units', 'predicted_units_rounded',
                      'tier', 'demand_class']])
    print(f'\n{"═" * 80}')
    print(f'  {r["nest"]}  |  {r["facility"]}  (top 10 predicted products)')
    print(f'{"═" * 80}')
    print(sub.to_string(index=False))


════════════════════════════════════════════════════════════════════════════════
  RW2 Kayonza  |  Bushara HC  (top 10 predicted products)
════════════════════════════════════════════════════════════════════════════════
                                 product  predicted_units  predicted_units_rounded       tier demand_class
  READY-TO-USE THERAPEUTIC FOODS(RUTF)92        10.869565                       11       rare Intermittent
             A.I CATHETER LARGE (V.A.F )         1.754987                        2 modellable Intermittent
                A.I CATHETER LARGE (MPB)         1.621782                        2 modellable        Lumpy
                        NOVOFINE 32G 4MM         1.594203                        2       rare Intermittent
      Landrace Pig Semen 100ml B/1 (VAF)         1.507005                        2 modellable Intermittent
               PPR VACCINE 100 DOSES B/1         1.275362                        1       rare Intermittent
       SOLVENT PPR VACCINE 100

### Step 12 — Summary Statistics of the Shopping List

In [19]:
print('Shopping List Summary')
print(f'{"─" * 60}')
print(f'Total (nest, facility, product) pairs: {len(shopping):,}')
print(f'Pairs with non-zero predicted demand:  '
      f'{(shopping["predicted_units_rounded"] > 0).sum():,}')
print(f'\nTotal predicted units by nest:')
print(shopping.groupby('nest')['predicted_units_rounded'].sum())
print(f'\nTotal predicted units by tier:')
print(shopping.groupby('tier')['predicted_units_rounded'].sum())
print(f'\nTotal predicted units by demand class:')
print(shopping.groupby('demand_class')['predicted_units_rounded'].sum())
print(f'\nTotal elapsed: {time.time()-t0:.1f}s')

Shopping List Summary
────────────────────────────────────────────────────────────
Total (nest, facility, product) pairs: 48,356
Pairs with non-zero predicted demand:  9,350

Total predicted units by nest:
nest
GH1 Omenako     1818
GH3 Vobsi       6030
RW1 Muhanga    22583
RW2 Kayonza    14876
Name: predicted_units_rounded, dtype: int64

Total predicted units by tier:
tier
modellable    20163
rare          25144
Name: predicted_units_rounded, dtype: int64

Total predicted units by demand class:
demand_class
Erratic            26
Intermittent    31716
Lumpy           12515
Smooth            277
Zero_only         773
Name: predicted_units_rounded, dtype: int64

Total elapsed: 373.9s


### Step 13 — Download the Shopping List

In [20]:
files.download(str(DATA_DIR / 'facility_shopping_list_next_week.csv'))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>